In [5]:
# Google Colab, Python 3.x 기준으로 작성한 노트북입니다.
# 이 셀은 전체 노트북에서 사용할 라이브러리를 한곳에 모아 둡니다.
# 과제에서 허용된 scikit-learn, pandas, numpy, joblib, pecab만 사용합니다.

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import matthews_corrcoef, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline




In [6]:
# MCC 최적 threshold를 final_pipeline.pkl 안에 함께 저장하기 위한 wrapper입니다.
# base_pipeline은 predict_proba를 제공하는 sklearn Pipeline이어야 합니다.
# predict()는 POSITIVE 확률이 threshold 이상이면 POSITIVE, 아니면 NEGATIVE를 반환합니다.
class ThresholdedPipeline(BaseEstimator, ClassifierMixin):
    def __init__(self, base_pipeline, threshold=0.5, pos_label="POSITIVE", neg_label="NEGATIVE"):
        self.base_pipeline = base_pipeline
        self.threshold = threshold
        self.pos_label = pos_label
        self.neg_label = neg_label

    def fit(self, X, y):
        self.base_pipeline_ = clone(self.base_pipeline)
        self.base_pipeline_.fit(X, y)
        self.classes_ = self.base_pipeline_.named_steps["classifier"].classes_
        return self

    def predict_proba(self, X):
        return self.base_pipeline_.predict_proba(X)

    def predict(self, X):
        proba = self.predict_proba(X)
        pos_idx = list(self.classes_).index(self.pos_label)
        pos_score = proba[:, pos_idx]
        return np.where(pos_score >= self.threshold, self.pos_label, self.neg_label)



In [7]:
# Colab에서 실행할 때는 Google Drive를 마운트해야 파일이 세션 종료 후에도 남습니다.
# 로컬 Jupyter 또는 VS Code에서 실행하는 경우 google.colab 모듈이 없으므로 예외 처리로 건너뜁니다.

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", type(exc).__name__)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# pecab 형태소 분석기를 준비합니다.
# Colab에서 pecab이 설치되어 있지 않다면 이 셀 위에 새 코드 셀을 만들고 다음 명령을 한 번 실행하세요.
%pip install -q pecab

try:
    from pecab import PeCab
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "pecab 패키지가 필요합니다. Colab에서 `%pip install -q pecab` 실행 후 이 셀부터 다시 실행하세요."
    ) from exc

pecab_tagger = PeCab()
print("pecab sample:", pecab_tagger.morphs("한국어 감성 분류 테스트 문장입니다."))



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 27.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 10.2 MB/s eta 0:00:00
pecab sample: ['한국어', '감성', '분류', '테스트', '문장', '입니다', '.']


In [9]:
# 실험 재현성을 위해 랜덤 시드를 하나로 고정합니다.
# random_state를 지원하는 StratifiedKFold, LogisticRegression, SGDClassifier에 같은 값을 사용합니다.
RANDOM_STATE = 42

# True이면 각 모델의 주요 하이퍼파라미터를 5-fold CV로 먼저 튜닝합니다.
# 전체 데이터와 텍스트 벡터화를 반복하므로 시간이 오래 걸릴 수 있습니다.
RUN_TUNING = True

# True이면 튜닝된 모델 3개를 묶은 soft voting의 가중치 후보도 5-fold CV로 비교합니다.
RUN_WEIGHT_TUNING = True

# True이면 OOF predict_proba로 MCC를 최대화하는 POSITIVE threshold를 찾습니다.
RUN_THRESHOLD_TUNING = True

# True이면 최종 soft voting 파이프라인 자체의 5-fold CV 결과를 남깁니다.
RUN_CV = False

# public_train.csv와 public_test.csv가 들어 있을 수 있는 위치를 순서대로 확인합니다.
# Colab에서는 보통 /content/drive/MyDrive/UD_26에 두면 됩니다.
# 만약 Drive 안에 mid 폴더까지 같이 올렸다면 /content/drive/MyDrive/UD_26/mid도 자동으로 찾습니다.
# 로컬에서는 이 노트북이 있는 저장소의 mid 폴더를 사용합니다.
DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/MyDrive/UD_26/mid"),
    Path("mid"),
    Path("."),
]

DATA_DIR = None
for candidate_dir in DATA_DIR_CANDIDATES:
    train_file = candidate_dir / "public_train.csv"
    test_file = candidate_dir / "public_test.csv"

    if train_file.exists() and test_file.exists():
        DATA_DIR = candidate_dir
        break

# 파일을 못 찾았을 때 현재 폴더로 조용히 넘어가지 않고, 확인해야 할 경로를 바로 보여줍니다.
if DATA_DIR is None:
    checked_paths = "\n".join(
        f"- {candidate_dir / 'public_train.csv'} and {candidate_dir / 'public_test.csv'}"
        for candidate_dir in DATA_DIR_CANDIDATES
    )
    raise FileNotFoundError(
        "public_train.csv와 public_test.csv를 찾지 못했습니다.\n"
        "다음 경로 중 한 곳에 두었는지 확인하세요:\n"
        f"{checked_paths}"
    )

# submission CSV is saved inside MODEL_DIR with the model artifacts.
# 모델 pkl 파일은 여러 실험을 비교하기 쉽도록 model/노트북이름 폴더에 따로 모읍니다.
OUTPUT_DIR = DATA_DIR
NOTEBOOK_TAG = "SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph"
MODEL_DIR = DATA_DIR / "model" / NOTEBOOK_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "public_train.csv"
TEST_PATH = DATA_DIR / "public_test.csv"

TUNING_RESULT_PATH = MODEL_DIR / "hyperparameter_tuning_results.csv"
WEIGHT_TUNING_RESULT_PATH = MODEL_DIR / "soft_voting_weight_tuning_results.csv"
FINAL_MODEL_PATH = MODEL_DIR / "final_pipeline.pkl"
SUBMISSION_PATH = MODEL_DIR / "submission.csv"

print("DATA_DIR:", DATA_DIR.resolve())
print("TRAIN_PATH:", TRAIN_PATH.resolve())
print("TEST_PATH:", TEST_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())




DATA_DIR: /content/drive/MyDrive/UD_26
TRAIN_PATH: /content/drive/MyDrive/UD_26/public_train.csv
TEST_PATH: /content/drive/MyDrive/UD_26/public_test.csv
OUTPUT_DIR: /content/drive/MyDrive/UD_26
MODEL_DIR: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph


In [10]:
# 학습 데이터와 테스트 데이터를 불러옵니다.
# public_train.csv에는 정답 라벨이 있고, public_test.csv에는 예측해야 할 text만 있습니다.
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# 제출 전 기본 스키마를 먼저 확인합니다.
# 컬럼명이 다르면 이후 셀이 조용히 틀린 결과를 만들 수 있으므로 assert로 바로 멈추게 합니다.
print("train shape:", train.shape)
print("test shape:", test.shape)
print("train columns:", train.columns.tolist())
print("test columns:", test.columns.tolist())
print("label distribution:")
print(train["label"].value_counts())

# 과제에서 요구한 라벨 문자열 POSITIVE / NEGATIVE를 그대로 사용합니다.
assert {"row_id", "text", "label"}.issubset(train.columns)
assert {"row_id", "text"}.issubset(test.columns)
assert set(train["label"].unique()) <= {"POSITIVE", "NEGATIVE"}
assert train["text"].notna().all()
assert test["text"].notna().all()



train shape: (149995, 3)
test shape: (49997, 2)
train columns: ['row_id', 'text', 'label']
test columns: ['row_id', 'text']
label distribution:
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64


In [11]:
# 모델 입력으로 사용할 raw text와 pecab 형태소 text를 함께 준비합니다.
# pecab_morph_text는 형태소를 공백으로 이어 붙인 문자열입니다.
# final_pipeline.pkl은 raw_text와 pecab_morph_text 두 컬럼이 있는 DataFrame을 입력으로 받습니다.
train_text = train["text"].astype(str)
test_text = test["text"].astype(str)

train_pecab_morph_text = [
    " ".join(pecab_tagger.morphs(text_value))
    for text_value in train_text
]
test_pecab_morph_text = [
    " ".join(pecab_tagger.morphs(text_value))
    for text_value in test_text
]

X = pd.DataFrame({
    "raw_text": train_text.values,
    "pecab_morph_text": train_pecab_morph_text,
})
y = train["label"].astype(str)

test_features = pd.DataFrame({
    "raw_text": test_text.values,
    "pecab_morph_text": test_pecab_morph_text,
})

print("train rows:", X.shape[0])
print("test rows:", test_features.shape[0])
print("feature columns:", X.columns.tolist())
print("label distribution:")
print(y.value_counts())
print("sample pecab_morph_text:")
print(X["pecab_morph_text"].head())



/usr/local/lib/python3.12/dist-packages/pecab/_tokenizer.py:265: RuntimeWarning: overflow encountered in scalar add
  from_pos_data.costs[idx]
/usr/local/lib/python3.12/dist-packages/pecab/_tokenizer.py:274: RuntimeWarning: overflow encountered in scalar add
  least_cost += word_cost


train rows: 149995
test rows: 49997
feature columns: ['raw_text', 'pecab_morph_text']
label distribution:
label
NEGATIVE    75170
POSITIVE    74825
Name: count, dtype: int64
sample pecab_morph_text:
0                                   아이 싱거 워라 . . . ㅠ ㅠ
1    히라이켄 의 히토미 오토 지테 노래 를 먼저 듣 고 알 게 되 어 보 게 되 었 는...
2                               올 해 본 영화 들 중 제일 맘 에 든다
3    어릴 적 로베르토 베니니 나온 실사 영화 를 재밌 게 봤었 는데 - 십 여 년 이 ...
4    주드 의 팬 이 라면 대략 어의 없 을 듯 . 사무엘 은 네 고 시 에 이터 의 변죽 ?
Name: pecab_morph_text, dtype: object


In [12]:
# 이 노트북의 실험 설계 원칙입니다.
# 튜닝할 값: 모델별 핵심 하이퍼파라미터와 soft voting 가중치입니다.
# 고정할 값: 벡터라이저 구조, TF-IDF 설정, solver/loss/penalty 같은 안정성 관련 설정입니다.
# 이유: 벡터라이저까지 전부 grid search하면 조합 수와 실행 시간이 너무 커지므로, 한국어 짧은 문장 분류에서 안정적인 preset으로 고정합니다.

# 고정 벡터라이저 preset입니다.
# raw text에서는 char, char_wb, word feature를 만들고, pecab_morph_text에서는 형태소 word n-gram feature를 만듭니다.
RAW_CHAR_NGRAM_RANGE = (2, 5)
RAW_CHAR_WB_NGRAM_RANGE = (2, 5)
RAW_WORD_NGRAM_RANGE = (1, 2)
PECAB_MORPH_NGRAM_RANGE = (1, 2)
MIN_DF = 2
RAW_CHAR_MAX_FEATURES = 400_000
RAW_CHAR_WB_MAX_FEATURES = 300_000
RAW_WORD_MAX_FEATURES = 200_000
PECAB_MORPH_MAX_FEATURES = 250_000
VECTORIZER_TAG = "rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k"

# 고정 TF-IDF preset입니다.
# sparse text feature와 선형 모델 조합에서 안정적인 기본값으로 둡니다.
TFIDF_SUBLINEAR_TF = True
TFIDF_NORM = "l2"
TFIDF_USE_IDF = True
TFIDF_SMOOTH_IDF = True

text_features = ColumnTransformer([
    ("raw_char", CountVectorizer(
        analyzer="char",
        ngram_range=RAW_CHAR_NGRAM_RANGE,
        min_df=MIN_DF,
        max_features=RAW_CHAR_MAX_FEATURES,
        lowercase=True,
        dtype=np.float32,
    ), "raw_text"),
    ("raw_char_wb", CountVectorizer(
        analyzer="char_wb",
        ngram_range=RAW_CHAR_WB_NGRAM_RANGE,
        min_df=MIN_DF,
        max_features=RAW_CHAR_WB_MAX_FEATURES,
        lowercase=True,
        dtype=np.float32,
    ), "raw_text"),
    ("raw_word", CountVectorizer(
        analyzer="word",
        ngram_range=RAW_WORD_NGRAM_RANGE,
        min_df=MIN_DF,
        max_features=RAW_WORD_MAX_FEATURES,
        token_pattern=r"(?u)\b\w+\b",
        lowercase=True,
        dtype=np.float32,
    ), "raw_text"),
    ("pecab_morph", CountVectorizer(
        analyzer="word",
        ngram_range=PECAB_MORPH_NGRAM_RANGE,
        min_df=MIN_DF,
        max_features=PECAB_MORPH_MAX_FEATURES,
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        dtype=np.float32,
    ), "pecab_morph_text"),
])

# 각 모델의 하이퍼파라미터 후보 범위입니다.
# 후보를 작게 잡아 실행 시간을 관리합니다.
LOGISTIC_C_VALUES = [0.5, 1.0, 2.0, 4.0]
NB_ALPHA_VALUES = [0.001, 0.005, 0.01, 0.05, 0.1]
SGD_ALPHA_VALUES = [0.00001, 0.00003, 0.0001, 0.0003, 0.001]

# soft voting 가중치 후보입니다.
# 순서는 LogisticRegression, ComplementNB, SGDClassifier입니다.
VOTING_WEIGHT_CANDIDATES = [
    (1.0, 1.0, 1.0),
    (1.5, 1.0, 1.0),
    (1.0, 1.5, 1.0),
    (1.0, 1.0, 1.5),
    (1.5, 1.5, 1.0),
    (1.5, 1.0, 1.5),
    (1.0, 1.5, 1.5),
]

# RUN_TUNING=False 또는 RUN_WEIGHT_TUNING=False로 빠르게 실행할 때 사용할 기본값입니다.
BEST_LOGISTIC_C = 2.0
BEST_NB_ALPHA = 0.01
BEST_SGD_ALPHA = 0.0001
BEST_VOTING_WEIGHTS = (1.0, 1.0, 1.0)
BEST_THRESHOLD = 0.5

print("fixed vectorizer preset:", VECTORIZER_TAG)
print("tuned LogisticRegression C candidates:", LOGISTIC_C_VALUES)
print("tuned ComplementNB alpha candidates:", NB_ALPHA_VALUES)
print("tuned SGDClassifier alpha candidates:", SGD_ALPHA_VALUES)
print("tuned soft voting weight candidates:", VOTING_WEIGHT_CANDIDATES)




fixed vectorizer preset: rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k
tuned LogisticRegression C candidates: [0.5, 1.0, 2.0, 4.0]
tuned ComplementNB alpha candidates: [0.001, 0.005, 0.01, 0.05, 0.1]
tuned SGDClassifier alpha candidates: [1e-05, 3e-05, 0.0001, 0.0003, 0.001]
tuned soft voting weight candidates: [(1.0, 1.0, 1.0), (1.5, 1.0, 1.0), (1.0, 1.5, 1.0), (1.0, 1.0, 1.5), (1.5, 1.5, 1.0), (1.5, 1.0, 1.5), (1.0, 1.5, 1.5)]


In [13]:
# 각 모델을 단독 Pipeline으로 구성해서 하이퍼파라미터 후보를 5-fold CV로 비교합니다.
# 튜닝도 전체 public_train.csv의 X, y를 사용하며, test 데이터는 사용하지 않습니다.
# 여기서 고른 best 값은 soft voting 앙상블에 들어갑니다.
mcc_scorer = make_scorer(matthews_corrcoef)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

tuning_rows = []

if RUN_TUNING:
    for c_value in LOGISTIC_C_VALUES:
        tuning_pipeline = Pipeline([
            ("features", text_features),
            ("tfidf", TfidfTransformer(
                sublinear_tf=TFIDF_SUBLINEAR_TF,
                norm=TFIDF_NORM,
                use_idf=TFIDF_USE_IDF,
                smooth_idf=TFIDF_SMOOTH_IDF,
            )),
            ("classifier", LogisticRegression(
                C=c_value,
                solver="liblinear",
                max_iter=1000,
                random_state=RANDOM_STATE,
            )),
        ])

        scores = cross_val_score(
            tuning_pipeline,
            X,
            y,
            scoring=mcc_scorer,
            cv=cv,
            n_jobs=-1,
        )

        tuning_rows.append({
            "model": "LogisticRegression",
            "parameter": "C",
            "value": c_value,
            "mean_mcc": scores.mean(),
            "std_mcc": scores.std(),
            "fold_scores": scores.tolist(),
        })
        print("LogisticRegression C=", c_value, "MCC mean=", scores.mean(), "std=", scores.std())

    for alpha_value in NB_ALPHA_VALUES:
        tuning_pipeline = Pipeline([
            ("features", text_features),
            ("tfidf", TfidfTransformer(
                sublinear_tf=TFIDF_SUBLINEAR_TF,
                norm=TFIDF_NORM,
                use_idf=TFIDF_USE_IDF,
                smooth_idf=TFIDF_SMOOTH_IDF,
            )),
            ("classifier", ComplementNB(
                alpha=alpha_value,
            )),
        ])

        scores = cross_val_score(
            tuning_pipeline,
            X,
            y,
            scoring=mcc_scorer,
            cv=cv,
            n_jobs=-1,
        )

        tuning_rows.append({
            "model": "ComplementNB",
            "parameter": "alpha",
            "value": alpha_value,
            "mean_mcc": scores.mean(),
            "std_mcc": scores.std(),
            "fold_scores": scores.tolist(),
        })
        print("ComplementNB alpha=", alpha_value, "MCC mean=", scores.mean(), "std=", scores.std())

    for alpha_value in SGD_ALPHA_VALUES:
        tuning_pipeline = Pipeline([
            ("features", text_features),
            ("tfidf", TfidfTransformer(
                sublinear_tf=TFIDF_SUBLINEAR_TF,
                norm=TFIDF_NORM,
                use_idf=TFIDF_USE_IDF,
                smooth_idf=TFIDF_SMOOTH_IDF,
            )),
            ("classifier", SGDClassifier(
                loss="log_loss",
                penalty="l2",
                alpha=alpha_value,
                max_iter=1000,
                tol=1e-3,
                random_state=RANDOM_STATE,
            )),
        ])

        scores = cross_val_score(
            tuning_pipeline,
            X,
            y,
            scoring=mcc_scorer,
            cv=cv,
            n_jobs=-1,
        )

        tuning_rows.append({
            "model": "SGDClassifier",
            "parameter": "alpha",
            "value": alpha_value,
            "mean_mcc": scores.mean(),
            "std_mcc": scores.std(),
            "fold_scores": scores.tolist(),
        })
        print("SGDClassifier alpha=", alpha_value, "MCC mean=", scores.mean(), "std=", scores.std())

    tuning_results = pd.DataFrame(tuning_rows)
    tuning_results = tuning_results.sort_values(
        ["model", "mean_mcc"],
        ascending=[True, False],
    ).reset_index(drop=True)

    BEST_LOGISTIC_C = float(
        tuning_results[tuning_results["model"] == "LogisticRegression"]
        .sort_values("mean_mcc", ascending=False)
        .iloc[0]["value"]
    )
    BEST_NB_ALPHA = float(
        tuning_results[tuning_results["model"] == "ComplementNB"]
        .sort_values("mean_mcc", ascending=False)
        .iloc[0]["value"]
    )
    BEST_SGD_ALPHA = float(
        tuning_results[tuning_results["model"] == "SGDClassifier"]
        .sort_values("mean_mcc", ascending=False)
        .iloc[0]["value"]
    )

    tuning_results.to_csv(TUNING_RESULT_PATH, index=False, encoding="utf-8")
    print("saved tuning results:", TUNING_RESULT_PATH)
else:
    tuning_results = pd.DataFrame([
        {"model": "LogisticRegression", "parameter": "C", "value": BEST_LOGISTIC_C},
        {"model": "ComplementNB", "parameter": "alpha", "value": BEST_NB_ALPHA},
        {"model": "SGDClassifier", "parameter": "alpha", "value": BEST_SGD_ALPHA},
    ])
    print("RUN_TUNING=False: 기본 하이퍼파라미터를 사용합니다.")

print("BEST_LOGISTIC_C:", BEST_LOGISTIC_C)
print("BEST_NB_ALPHA:", BEST_NB_ALPHA)
print("BEST_SGD_ALPHA:", BEST_SGD_ALPHA)
print(tuning_results)



LogisticRegression C= 0.5 MCC mean= 0.7284833687857694 std= 0.005380985023317932
LogisticRegression C= 1.0 MCC mean= 0.7410769142106457 std= 0.0046724170233955736
LogisticRegression C= 2.0 MCC mean= 0.7486317249556744 std= 0.003795878377130961
LogisticRegression C= 4.0 MCC mean= 0.7519110406809777 std= 0.004266338359237206
ComplementNB alpha= 0.001 MCC mean= 0.7444486919981435 std= 0.001750077620821344
ComplementNB alpha= 0.005 MCC mean= 0.7453426113079223 std= 0.0028131902589900127
ComplementNB alpha= 0.01 MCC mean= 0.746361126314125 std= 0.0036110615328681208
ComplementNB alpha= 0.05 MCC mean= 0.7457213053687493 std= 0.0035019378561850193
ComplementNB alpha= 0.1 MCC mean= 0.744041552972719 std= 0.0042005126568066
SGDClassifier alpha= 1e-05 MCC mean= 0.7384842378876337 std= 0.004804192504787617
SGDClassifier alpha= 3e-05 MCC mean= 0.7152585090194379 std= 0.005008383762341898
SGDClassifier alpha= 0.0001 MCC mean= 0.6810165574235405 std= 0.006051899268054761
SGDClassifier alpha= 0.0003 

In [14]:
# 튜닝된 세 모델을 사용해 soft voting 가중치 후보를 5-fold CV로 비교합니다.
# soft voting은 각 모델의 predict_proba 값을 평균하므로 hard voting보다 가중치 의미가 자연스럽습니다.
weight_rows = []

if RUN_WEIGHT_TUNING:
    for weight_values in VOTING_WEIGHT_CANDIDATES:
        classifier = VotingClassifier(
            estimators=[
                ("logistic_tuned", LogisticRegression(
                    C=BEST_LOGISTIC_C,
                    solver="liblinear",
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                )),
                ("complementnb_tuned", ComplementNB(
                    alpha=BEST_NB_ALPHA,
                )),
                ("sgd_log_loss_tuned", SGDClassifier(
                    loss="log_loss",
                    penalty="l2",
                    alpha=BEST_SGD_ALPHA,
                    max_iter=1000,
                    tol=1e-3,
                    random_state=RANDOM_STATE,
                )),
            ],
            voting="soft",
            weights=weight_values,
        )

        weight_pipeline = Pipeline([
            ("features", text_features),
            ("tfidf", TfidfTransformer(
                sublinear_tf=TFIDF_SUBLINEAR_TF,
                norm=TFIDF_NORM,
                use_idf=TFIDF_USE_IDF,
                smooth_idf=TFIDF_SMOOTH_IDF,
            )),
            ("classifier", classifier),
        ])

        scores = cross_val_score(
            weight_pipeline,
            X,
            y,
            scoring=mcc_scorer,
            cv=cv,
            n_jobs=-1,
        )

        weight_rows.append({
            "weights": weight_values,
            "logistic_weight": weight_values[0],
            "complementnb_weight": weight_values[1],
            "sgd_weight": weight_values[2],
            "mean_mcc": scores.mean(),
            "std_mcc": scores.std(),
            "fold_scores": scores.tolist(),
        })
        print("weights=", weight_values, "MCC mean=", scores.mean(), "std=", scores.std())

    weight_tuning_results = pd.DataFrame(weight_rows)
    weight_tuning_results = weight_tuning_results.sort_values(
        "mean_mcc",
        ascending=False,
    ).reset_index(drop=True)

    BEST_VOTING_WEIGHTS = tuple(weight_tuning_results.iloc[0]["weights"])
    weight_tuning_results.to_csv(WEIGHT_TUNING_RESULT_PATH, index=False, encoding="utf-8")
    print("saved weight tuning results:", WEIGHT_TUNING_RESULT_PATH)
else:
    weight_tuning_results = pd.DataFrame([
        {
            "weights": BEST_VOTING_WEIGHTS,
            "logistic_weight": BEST_VOTING_WEIGHTS[0],
            "complementnb_weight": BEST_VOTING_WEIGHTS[1],
            "sgd_weight": BEST_VOTING_WEIGHTS[2],
        }
    ])
    print("RUN_WEIGHT_TUNING=False: 기본 soft voting 가중치를 사용합니다.")

print("BEST_VOTING_WEIGHTS:", BEST_VOTING_WEIGHTS)
print(weight_tuning_results)



weights= (1.0, 1.0, 1.0) MCC mean= 0.7563155788686557 std= 0.0027208551021969864
weights= (1.5, 1.0, 1.0) MCC mean= 0.7574899085778 std= 0.0025858605460146906
weights= (1.0, 1.5, 1.0) MCC mean= 0.7550490354587656 std= 0.00284787075379889
weights= (1.0, 1.0, 1.5) MCC mean= 0.7561278475128308 std= 0.0029003193690587694
weights= (1.5, 1.5, 1.0) MCC mean= 0.7565441972886084 std= 0.0024001114462705976
weights= (1.5, 1.0, 1.5) MCC mean= 0.7576008318750372 std= 0.0029578110173028465


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


weights= (1.0, 1.5, 1.5) MCC mean= 0.7557272539064448 std= 0.0026341792795944396
saved weight tuning results: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/soft_voting_weight_tuning_results.csv
BEST_VOTING_WEIGHTS: (1.5, 1.0, 1.5)
           weights  logistic_weight  complementnb_weight  sgd_weight  \
0  (1.5, 1.0, 1.5)              1.5                  1.0         1.5   
1  (1.5, 1.0, 1.0)              1.5                  1.0         1.0   
2  (1.5, 1.5, 1.0)              1.5                  1.5         1.0   
3  (1.0, 1.0, 1.0)              1.0                  1.0         1.0   
4  (1.0, 1.0, 1.5)              1.0                  1.0         1.5   
5  (1.0, 1.5, 1.5)              1.0                  1.5         1.5   
6  (1.0, 1.5, 1.0)              1.0                  1.5         1.0   

   mean_mcc   std_mcc                                        fold_scores  
0  0.757601  0.002958  [0.7598567960297018, 0.7532582071981496, 0.754

In [15]:
# 튜닝에서 선택된 하이퍼파라미터와 soft voting 가중치로 최종 앙상블을 구성합니다.
# 세 모델 모두 predict_proba를 제공하므로 VotingClassifier의 voting="soft"를 사용합니다.
classifier = VotingClassifier(
    estimators=[
        ("logistic_tuned", LogisticRegression(
            C=BEST_LOGISTIC_C,
            solver="liblinear",
            max_iter=1000,
            random_state=RANDOM_STATE,
        )),
        ("complementnb_tuned", ComplementNB(
            alpha=BEST_NB_ALPHA,
        )),
        ("sgd_log_loss_tuned", SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=BEST_SGD_ALPHA,
            max_iter=1000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        )),
    ],
    voting="soft",
    weights=BEST_VOTING_WEIGHTS,
)

pipeline = Pipeline([
    ("features", text_features),
    ("tfidf", TfidfTransformer(
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        norm=TFIDF_NORM,
        use_idf=TFIDF_USE_IDF,
        smooth_idf=TFIDF_SMOOTH_IDF,
    )),
    ("classifier", classifier),
])

logistic_c_tag = str(BEST_LOGISTIC_C).replace(".", "p")
nb_alpha_tag = str(BEST_NB_ALPHA).replace(".", "p")
sgd_alpha_tag = str(BEST_SGD_ALPHA).replace(".", "p")
weight_tag = "w" + "_".join(str(x).replace(".", "p") for x in BEST_VOTING_WEIGHTS)

EXPERIMENT_TAG = (
    f"softvoting_logreg_C{logistic_c_tag}_"
    f"complementnb_alpha{nb_alpha_tag}_"
    f"sgdlogloss_alpha{sgd_alpha_tag}_"
    f"{weight_tag}_{VECTORIZER_TAG}_tfidf_rs42"
)
MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_TAG}.pkl"

print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("MODEL_PATH:", MODEL_PATH)
pipeline



EXPERIMENT_TAG: softvoting_logreg_C4p0_complementnb_alpha0p01_sgdlogloss_alpha1e-05_w1p5_1p0_1p5_rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k_tfidf_rs42
MODEL_PATH: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/softvoting_logreg_C4p0_complementnb_alpha0p01_sgdlogloss_alpha1e-05_w1p5_1p0_1p5_rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k_tfidf_rs42.pkl


Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('raw_char',
                                                  CountVectorizer(analyzer='char',
                                                                  dtype=<class 'numpy.float32'>,
                                                                  max_features=400000,
                                                                  min_df=2,
                                                                  ngram_range=(2,
                                                                               5)),
                                                  'raw_text'),
                                                 ('raw_char_wb',
                                                  CountVectorizer(analyzer='char_wb',
                                                                  dtype=<class 'numpy.float32'>,
                                                                  max_features=300000,
                                                                  min_df=2,
                                                                  ngram_range=(2,
                                                                               5)),
                                                  'raw_text'),
                                                 ('raw_word',
                                                  CountV...
                ('tfidf', TfidfTransformer(sublinear_tf=True)),
                ('classifier',
                 VotingClassifier(estimators=[('logistic_tuned',
                                               LogisticRegression(C=4.0,
                                                                  max_iter=1000,
                                                                  random_state=42,
                                                                  solver='liblinear')),
                                              ('complementnb_tuned',
                                               ComplementNB(alpha=0.01)),
                                              ('sgd_log_loss_tuned',
                                               SGDClassifier(alpha=1e-05,
                                                             loss='log_loss',
                                                             random_state=42))],
                                  voting='soft', weights=(1.5, 1.0, 1.5)))])

In [16]:
# 최종 soft voting 파이프라인 자체를 전체 public_train.csv 기준 5-fold CV로 평가합니다.
# 이 점수가 최종 앙상블 구성의 대표 CV MCC입니다.
if RUN_CV:
    cv_scores = cross_val_score(
        pipeline,
        X,
        y,
        scoring=mcc_scorer,
        cv=cv,
        n_jobs=-1,
    )

    print("5-fold CV MCC scores:", cv_scores)
    print("5-fold CV MCC mean:", cv_scores.mean())
    print("5-fold CV MCC std:", cv_scores.std())
else:
    print("RUN_CV=False: skipped final ensemble 5-fold CV.")



RUN_CV=False: skipped final ensemble 5-fold CV.


In [17]:
# 최종 soft voting 파이프라인의 OOF predict_proba로 MCC를 최대화하는 threshold를 찾습니다.
# 각 train row의 score는 자신이 속하지 않은 fold에서 학습된 모델로 예측된 확률입니다.
# 이렇게 찾은 BEST_THRESHOLD는 ThresholdedPipeline에 저장되어 final_pipeline.predict에도 반영됩니다.
if RUN_THRESHOLD_TUNING:
    oof_proba = cross_val_predict(
        pipeline,
        X,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )

    pos_idx = list(np.unique(y)).index("POSITIVE")
    oof_pos_score = np.asarray(oof_proba[:, pos_idx], dtype=float)
    y_true_array = np.asarray(y)
    y_pos = (y_true_array == "POSITIVE").astype(int)

    order = np.argsort(oof_pos_score)
    score_sorted = oof_pos_score[order]
    y_sorted = y_pos[order]
    n_scores = len(score_sorted)

    change_points = np.where(score_sorted[1:] != score_sorted[:-1])[0] + 1
    candidate_k = np.r_[0, change_points, n_scores]

    cum_pos = np.r_[0, np.cumsum(y_sorted)]
    cum_neg = np.r_[0, np.cumsum(1 - y_sorted)]
    total_pos = cum_pos[-1]
    total_neg = cum_neg[-1]

    best_threshold_mcc = -999.0
    BEST_THRESHOLD = 0.5

    for k in candidate_k:
        tp = total_pos - cum_pos[k]
        fn = cum_pos[k]
        fp = total_neg - cum_neg[k]
        tn = cum_neg[k]

        denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        if denom == 0:
            mcc = 0.0
        else:
            mcc = (tp * tn - fp * fn) / denom

        if mcc > best_threshold_mcc:
            best_threshold_mcc = float(mcc)
            if k == 0:
                BEST_THRESHOLD = float(score_sorted[0] - 1e-12)
            elif k == n_scores:
                BEST_THRESHOLD = float(score_sorted[-1] + 1e-12)
            else:
                BEST_THRESHOLD = float((score_sorted[k - 1] + score_sorted[k]) / 2)

    oof_pred = np.where(oof_pos_score >= BEST_THRESHOLD, "POSITIVE", "NEGATIVE")
    print("BEST_THRESHOLD:", BEST_THRESHOLD)
    print("OOF threshold-tuned MCC:", matthews_corrcoef(y, oof_pred))
    print("OOF threshold prediction distribution:")
    print(pd.Series(oof_pred).value_counts())
else:
    print("RUN_THRESHOLD_TUNING=False: BEST_THRESHOLD=0.5를 사용합니다.")

threshold_tag = str(round(BEST_THRESHOLD, 6)).replace(".", "p").replace("-", "m")
EXPERIMENT_TAG = f"{EXPERIMENT_TAG}_thr{threshold_tag}"
MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_TAG}.pkl"
print("thresholded EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("thresholded MODEL_PATH:", MODEL_PATH)



/tmp/ipykernel_83710/3527975353.py:41: RuntimeWarning: overflow encountered in scalar multiply
  denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
/tmp/ipykernel_83710/3527975353.py:41: RuntimeWarning: invalid value encountered in sqrt
  denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))


BEST_THRESHOLD: 0.9459154862178671
OOF threshold-tuned MCC: 0.45037223085256145
OOF threshold prediction distribution:
NEGATIVE    123422
POSITIVE     26573
Name: count, dtype: int64
thresholded EXPERIMENT_TAG: softvoting_logreg_C4p0_complementnb_alpha0p01_sgdlogloss_alpha1e-05_w1p5_1p0_1p5_rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k_tfidf_rs42_thr0p945915
thresholded MODEL_PATH: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/softvoting_logreg_C4p0_complementnb_alpha0p01_sgdlogloss_alpha1e-05_w1p5_1p0_1p5_rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k_tfidf_rs42_thr0p945915.pkl


In [18]:
# 5-fold CV와 threshold 확인이 끝나면 전체 학습 데이터로 최종 모델을 학습합니다.
# base_final_pipeline은 predict_proba를 만들고, ThresholdedPipeline은 BEST_THRESHOLD를 predict에 반영합니다.
base_final_pipeline = Pipeline([
    ("features", text_features),
    ("tfidf", TfidfTransformer(
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        norm=TFIDF_NORM,
        use_idf=TFIDF_USE_IDF,
        smooth_idf=TFIDF_SMOOTH_IDF,
    )),
    ("classifier", classifier),
])

final_pipeline = ThresholdedPipeline(
    base_pipeline=base_final_pipeline,
    threshold=BEST_THRESHOLD,
)
final_pipeline.fit(X, y)

# 실험 파일명 모델과 과제 필수 파일명 final_pipeline.pkl을 둘 다 MODEL_DIR에 저장합니다.
# 이 final_pipeline.pkl은 raw_text와 pecab_morph_text 두 컬럼을 입력으로 받고, predict에는 BEST_THRESHOLD가 반영됩니다.
joblib.dump(final_pipeline, MODEL_PATH)
joblib.dump(final_pipeline, FINAL_MODEL_PATH)

print("BEST_THRESHOLD:", BEST_THRESHOLD)
print("saved experiment model:", MODEL_PATH)
print("saved final model:", FINAL_MODEL_PATH)



BEST_THRESHOLD: 0.9459154862178671
saved experiment model: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/softvoting_logreg_C4p0_complementnb_alpha0p01_sgdlogloss_alpha1e-05_w1p5_1p0_1p5_rawchar2to5_rawcharwb2to5_rawword1to2_pecabmorph1to2_mindf2_maxfeat400k300k200k250k_tfidf_rs42_thr0p945915.pkl
saved final model: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/final_pipeline.pkl


In [19]:
# public_test.csv의 text만 사용해 예측합니다.
# 예측 전 pecab_morph_text를 만든 뒤 final_pipeline에 raw_text와 함께 넣습니다.
test_pred = final_pipeline.predict(test_features)

# 제출 파일은 정확히 row_id, pred_label 두 컬럼만 가져야 합니다.
# row_id 순서는 public_test.csv의 순서를 그대로 유지합니다.
submission = test[["row_id"]].copy()
submission["pred_label"] = test_pred

# 제출 전 필수 조건을 assert로 확인합니다.
assert len(submission) == len(test)
assert submission.columns.tolist() == ["row_id", "pred_label"]
assert submission["row_id"].equals(test["row_id"])
assert set(submission["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}

submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print(submission.head())
print("Prediction distribution:")
print(submission["pred_label"].value_counts())
print("saved submission:", SUBMISSION_PATH)



        row_id pred_label
0  test-000001   POSITIVE
1  test-000002   NEGATIVE
2  test-000003   NEGATIVE
3  test-000004   NEGATIVE
4  test-000005   NEGATIVE
Prediction distribution:
pred_label
NEGATIVE    41082
POSITIVE     8915
Name: count, dtype: int64
saved submission: /content/drive/MyDrive/UD_26/submission.csv


In [20]:
# 저장한 final_pipeline.pkl을 다시 불러와서 실제 제출 artifact가 정상인지 간단히 확인합니다.
# 이 모델은 raw_text와 pecab_morph_text 두 컬럼이 있는 DataFrame을 입력으로 받습니다.
loaded_pipeline = joblib.load(FINAL_MODEL_PATH)
smoke_pred = loaded_pipeline.predict(test_features.head(5))

print("loaded model type:", type(loaded_pipeline))
print("smoke predictions:", smoke_pred)
assert set(smoke_pred) <= {"POSITIVE", "NEGATIVE"}



loaded model type: <class '__main__.ThresholdedPipeline'>
smoke predictions: ['POSITIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE']


In [21]:
# 제출 전 최종 검증 셀입니다.
# submission.csv와 threshold가 포함된 final_pipeline.pkl이 과제 요구사항에 맞게 생성됐는지 한 번에 확인합니다.
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("SUBMISSION_PATH:", SUBMISSION_PATH)
print("FINAL_MODEL_PATH:", FINAL_MODEL_PATH)
print("BEST_THRESHOLD:", BEST_THRESHOLD)

assert TRAIN_PATH.exists(), f"train file not found: {TRAIN_PATH}"
assert TEST_PATH.exists(), f"test file not found: {TEST_PATH}"
assert SUBMISSION_PATH.exists(), f"submission.csv not found: {SUBMISSION_PATH}"
assert FINAL_MODEL_PATH.exists(), f"final_pipeline.pkl not found: {FINAL_MODEL_PATH}"

submission_check = pd.read_csv(SUBMISSION_PATH)

print("submission shape:", submission_check.shape)
print("submission columns:", submission_check.columns.tolist())
print("submission pred_label distribution:")
print(submission_check["pred_label"].value_counts())

assert submission_check.columns.tolist() == ["row_id", "pred_label"]
assert len(submission_check) == len(test)
assert submission_check["row_id"].equals(test["row_id"])
assert set(submission_check["pred_label"].unique()) <= {"POSITIVE", "NEGATIVE"}
assert submission_check["row_id"].notna().all()
assert submission_check["pred_label"].notna().all()

loaded_final_pipeline = joblib.load(FINAL_MODEL_PATH)
sample_pred = loaded_final_pipeline.predict(test_features.head(10))
loaded_pred = loaded_final_pipeline.predict(test_features)

print("loaded model type:", type(loaded_final_pipeline))
print("loaded threshold:", loaded_final_pipeline.threshold)
print("sample predictions:", sample_pred)
print("loaded prediction distribution:")
print(pd.Series(loaded_pred).value_counts())

assert set(sample_pred) <= {"POSITIVE", "NEGATIVE"}
assert np.array_equal(loaded_pred, test_pred)
assert np.array_equal(loaded_pred, submission_check["pred_label"].values)

if "MODEL_PATH" in globals():
    print("MODEL_PATH:", MODEL_PATH)
    assert MODEL_PATH.exists(), f"experiment model file not found: {MODEL_PATH}"

if RUN_TUNING:
    print("TUNING_RESULT_PATH:", TUNING_RESULT_PATH)
    assert TUNING_RESULT_PATH.exists(), f"tuning result file not found: {TUNING_RESULT_PATH}"

if RUN_WEIGHT_TUNING:
    print("WEIGHT_TUNING_RESULT_PATH:", WEIGHT_TUNING_RESULT_PATH)
    assert WEIGHT_TUNING_RESULT_PATH.exists(), f"weight tuning result file not found: {WEIGHT_TUNING_RESULT_PATH}"

print("최종 제출 검증 완료")



TRAIN_PATH: /content/drive/MyDrive/UD_26/public_train.csv
TEST_PATH: /content/drive/MyDrive/UD_26/public_test.csv
SUBMISSION_PATH: /content/drive/MyDrive/UD_26/submission.csv
FINAL_MODEL_PATH: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/final_pipeline.pkl
BEST_THRESHOLD: 0.9459154862178671
submission shape: (49997, 2)
submission columns: ['row_id', 'pred_label']
submission pred_label distribution:
pred_label
NEGATIVE    41082
POSITIVE     8915
Name: count, dtype: int64
loaded model type: <class '__main__.ThresholdedPipeline'>
loaded threshold: 0.9459154862178671
sample predictions: ['POSITIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE' 'POSITIVE'
 'NEGATIVE' 'NEGATIVE' 'NEGATIVE' 'NEGATIVE']
loaded prediction distribution:
NEGATIVE    41082
POSITIVE     8915
Name: count, dtype: int64
MODEL_PATH: /content/drive/MyDrive/UD_26/model/SoftVoting_LogisticRegression_CompNB_SGDLogLoss_PecabMorph/softvoting_logreg_C4p0_complementnb_alpha0p01_sg